# Optional Challenge: Tricky Messages

We retrain each model with the best settings from Parts A, B and C on the same train.csv, then test the 3 tricky messages. All 3 should be ham (0).

In [2]:
%pip install numpy pandas scikit-learn gensim torch transformers


  Using cached setuptools-84.2.1-py3-none-any.whl.metadata (6.6 kB)
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
  Using cached networkx-3.7-py3-none-any.whl.metadata (6.7 kB)
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached fsspec-2026.9.0-py3-none-any.whl.metadata (10 kB)
  Using cached pyyaml-6.0.3-cp312-cp312-win_amd64.whl.metadata (2.4 kB)
  Using cached tqdm-4.70.1-py3-none-any.whl.metadata (57 kB)
  Using cached mpmath-1.3.0-py3-none-any.whl.metadata (8.6 kB)
  Using cached markupsafe-3.0.4-cp312-cp312-win_amd64.whl.metadata (2.8 kB)
  Using cached idna-3.20-py3-none-any.whl.metadata (7.2 kB)
  Using cached h11-0.16.0-py3-none-any.whl.metadata (8.3 kB)
   ---------------------------------------- 0.0/126.7 MB ? eta -:--:--
    --------------------------------------- 3.1/126.7 MB 30.7 MB/s eta 0:00:05
   ---- ----------------------------------- 14.4/126.7 MB 39.3 MB/s eta 0:00:03
   ----------- ---------------------------- 35.9/126.7 M


[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
import numpy as np
import pandas as pd
import torch
import gensim.downloader as api
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import KNeighborsClassifier
from transformers import BertTokenizer, BertModel

train_df = pd.read_csv("train.csv")
X_train = train_df["text"].astype(str)
y_train = train_df["label"]

tricky = [
    "You did not win a prize.",
    "Free coffee at the seminar today.",
    "Call this number when you can."
]

# print the labels of the k neighbors and the closest training message
def show_neighbors(knn, X_tricky):
    idx = knn.kneighbors(X_tricky, return_distance=False)
    for msg, row in zip(tricky, idx):
        print(msg)
        print("  neighbor labels:", list(y_train[row]))
        print("  closest message:", X_train[row[0]])

c:\Users\jason\Downloads\INSY669 Moudle Case 4 Github\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## TF-IDF + KNN (same as Part A)

In [4]:
tfidf = TfidfVectorizer(ngram_range=(1, 1))
X_train_tfidf = tfidf.fit_transform(X_train)
X_tricky_tfidf = tfidf.transform(tricky)

knn_tfidf = KNeighborsClassifier(n_neighbors=1, metric="cosine", algorithm="brute")
knn_tfidf.fit(X_train_tfidf, y_train)
pred_tfidf = knn_tfidf.predict(X_tricky_tfidf)

show_neighbors(knn_tfidf, X_tricky_tfidf)

You did not win a prize.
  neighbor labels: [1]
  closest message: Win a å£1000 cash prize or a prize worth å£5000
Free coffee at the seminar today.
  neighbor labels: [0]
  closest message: Coffee cake, i guess...
Call this number when you can.
  neighbor labels: [0]
  closest message: U can call now...


## Word2Vec avg + KNN ( same as Part B)

In [5]:
w2v = api.load("word2vec-google-news-300")

def average_word_vectors(text, model):
    words = text.split()
    vectors = [model[word] for word in words if word in model]
    if len(vectors) == 0:
        return np.zeros(model.vector_size)
    return np.mean(vectors, axis=0)

X_train_w2v = np.array([average_word_vectors(t, w2v) for t in X_train])
X_tricky_w2v = np.array([average_word_vectors(t, w2v) for t in tricky])
del w2v  # free memory before loading BERT

knn_w2v = KNeighborsClassifier(n_neighbors=3, weights="distance", metric="cosine")
knn_w2v.fit(X_train_w2v, y_train)
pred_w2v = knn_w2v.predict(X_tricky_w2v)

show_neighbors(knn_w2v, X_tricky_w2v)

You did not win a prize.
  neighbor labels: [0, 0, 0]
  closest message: Oh k.k..where did you take test?
Free coffee at the seminar today.
  neighbor labels: [1, 0, 0]
  closest message: Free Msg: Ringtone!From: http://tms. widelive.com/index. wml?id=1b6a5ecef91ff9*37819&first=true18:0430-JUL-05
Call this number when you can.
  neighbor labels: [1, 0, 0]
  closest message: You have 1 new message. Call 0207-083-6089


## BERT Sentence Embeddings + KNN （Same as part C）

In [6]:
device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")
bert_model = BertModel.from_pretrained("bert-base-uncased").to(device).eval()

def get_bert_embedding(texts):
    inputs = tokenizer(texts, return_tensors="pt", padding=True, truncation=True, max_length=128).to(device)
    with torch.no_grad():
        outputs = bert_model(**inputs)
    return outputs.last_hidden_state[:, 0, :].cpu().numpy()  # CLS token

def embed_all(texts, batch_size=64):
    batches = []
    for start in range(0, len(texts), batch_size):
        batches.append(get_bert_embedding(texts[start:start + batch_size]))
    return np.vstack(batches)

X_train_bert = embed_all(list(X_train))
X_tricky_bert = embed_all(tricky)

knn_bert = KNeighborsClassifier(n_neighbors=5, metric="cosine", algorithm="brute")
knn_bert.fit(X_train_bert, y_train)
pred_bert = knn_bert.predict(X_tricky_bert)

show_neighbors(knn_bert, X_tricky_bert)

c:\Users\jason\Downloads\INSY669 Moudle Case 4 Github\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\jason\.cache\huggingface\hub\models--bert-base-uncased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\jason\Downloads\INSY669 Moudle Case 4 Github\.venv\Lib\site-packages\huggingface_hu

You did not win a prize.
  neighbor labels: [0, 0, 0, 0, 0]
  closest message: You've always been the brainy one.
Free coffee at the seminar today.
  neighbor labels: [0, 0, 0, 0, 0]
  closest message: Going to join tomorrow.
Call this number when you can.
  neighbor labels: [0, 0, 0, 0, 0]
  closest message: When you get free, call me


## Results (1 = spam, 0 = ham)

In [7]:
results = pd.DataFrame({
    "Message": tricky,
    "TF-IDF": pred_tfidf,
    "Word2Vec": pred_w2v,
    "BERT": pred_bert
})
results

,Message,TF-IDF,Word2Vec,BERT
0,You did not win a prize.,1,0,0
1,Free coffee at the seminar today.,0,0,0
2,Call this number when you can.,0,0,0


## Compare All Three Models

BERT handled the tricky messages best. It classified all 3 messages as ham, and all 5 of its neighbors were ham for each message. Its closest matches were ordinary conversations, such as "When you get free, call me" for the ambiguous message.

TF-IDF misclassified the negation message as spam. Since k = 1, the prediction depended on a single neighbor, which was a spam message ("Win a ... cash prize or a prize worth ...") that also contains the words "win" and "prize". The other two messages were classified correctly because their closest messages were ham.

Word2Vec also classified all 3 messages as ham, but its neighbors were mixed. For the marketing and ambiguous messages, the closest neighbor was spam ("Free Msg: Ringtone!..." and "You have 1 new message. Call 0207-083-6089"), while the other 2 neighbors were ham.

In conlcusion, BERT and Word2Vec both matched the expected ham labels for all three messages. TF-IDF misclassified the negation message because its nearest neighbor was spam containing “win” and “prize.” BERT’s neighbors were all ham, while Word2Vec’s were mixed for two messages. However, both models got 3/3 correct, so this small test does not show that one is better overall.